# Chapter 5 — Numerical algorithms (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_python.ipynb)

Executable notebook of the chapter “Numerical algorithms”: it contains the algorithms `ChangeOfBase`, `BasicPrimalityTest`, `PrimalityTest`, `SieveOfEratosthenes` and `EuclideanAlgorithm` written in Python and run on the same examples as the lecture notes, printing the results and, where the notes show a trace, the step-by-step trace in the same format.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/algorithms/basic-algorithms/05-numerical-algorithms/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_java.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. They are plain Python cells, nothing to install; locally any Python 3 will do. After each algorithm there is a “Try it yourself” cell with an input to change.

**Indices.** In the pseudo-code arrays start from 1, here from 0: the element $a[i]$ of the pseudo-code is `a[i-1]` in the code. The printed traces use the indices of the code (as in the C++ examples of the lecture notes), so they are those of the pseudo-code with every index decreased by 1. The function names are those of the pseudo-code, in the Python convention (`snake_case`: `find_maximum` for `FindMaximum`).

## ChangeOfBase

**Input:** a natural number $n \ge 1$ (in base 10) and a base $b \ge 2$. **Output:** the array $d[1..nd]$ of the digits of $n$ in base $b$ and the number of digits $nd$.

One division by $b$ extracts the least significant digit ($r = n \bmod b$) and leaves as quotient the number written with the other digits: the division is repeated until the quotient is 0. The digits come out from the least significant to the most significant, so in the array `d` the digit of weight $b^0$ is in `d[0]` (in $d[1]$ in the pseudo-code).

The number of digits $nd = \lfloor \log_b n \rfloor + 1$ is computed here with integer multiplications only, without floating point logarithms: in double precision `log(1000)/log(10)` evaluates to $2.9999999999999996$ and the formula would give 3 digits instead of 4.

In [1]:
def change_of_base(n, b, trace=False):
    nd = 1                              # nd = ⌊log_b n⌋ + 1, with integer operations only
    p = b
    while p <= n:
        p = p * b
        nd = nd + 1
    d = [0] * nd                        # array of the digits
    k = 0                               # k = 1 in the pseudo-code
    while n > 0:
        q = n // b                      # quotient
        r = n % b                       # remainder (digit in base b)
        d[k] = r
        n = q
        k = k + 1
        if trace:
            print("%10d %5d %3d %5d %3d  %s" % (k, q, r, n, k, d[:k]))
    return d, nd


### Execution on the examples of the chapter

In [2]:
def digit_string(d, nd):                # the digits from the most to the least significant
    return "".join("0123456789ABCDEF"[d[i]] for i in range(nd - 1, -1, -1))


def run(n, b):
    print("Conversion: n = %d, b = %d" % (n, b))
    print("%10s %5s %3s %5s %3s  %s" % ("iteration", "q", "r", "n", "nd", "digits"))
    d, nd = change_of_base(n, b, trace=True)
    print("Return: nd = %d, d = %s" % (nd, d))
    print("%d = (%s)_%d" % (n, digit_string(d, nd), b))
    print()


run(7, 2)
run(8, 2)
run(25, 3)
run(100, 8)
run(1000, 10)    # 4 digits: the case where the floating point log(1000)/log(10) fails

Conversion: n = 7, b = 2
 iteration     q   r     n  nd  digits
         1     3   1     3   1  [1]
         2     1   1     1   2  [1, 1]
         3     0   1     0   3  [1, 1, 1]
Return: nd = 3, d = [1, 1, 1]
7 = (111)_2

Conversion: n = 8, b = 2
 iteration     q   r     n  nd  digits
         1     4   0     4   1  [0]
         2     2   0     2   2  [0, 0]
         3     1   0     1   3  [0, 0, 0]
         4     0   1     0   4  [0, 0, 0, 1]
Return: nd = 4, d = [0, 0, 0, 1]
8 = (1000)_2

Conversion: n = 25, b = 3
 iteration     q   r     n  nd  digits
         1     8   1     8   1  [1]
         2     2   2     2   2  [1, 2]
         3     0   2     0   3  [1, 2, 2]
Return: nd = 3, d = [1, 2, 2]
25 = (221)_3

Conversion: n = 100, b = 8
 iteration     q   r     n  nd  digits
         1    12   4    12   1  [4]
         2     1   4     1   2  [4, 4]
         3     0   1     0   3  [4, 4, 1]
Return: nd = 3, d = [4, 4, 1]
100 = (144)_8

Conversion: n = 1000, b = 10
 iteration     q   r

### Try it yourself

Change the input in the cell below and run it again. The numbers are $n$ and the base $b$ (up to 16: the digits above 9 are written A, B, …).

In [3]:
run(255, 16)

Conversion: n = 255, b = 16
 iteration     q   r     n  nd  digits
         1    15  15    15   1  [15]
         2     0  15     0   2  [15, 15]
Return: nd = 2, d = [15, 15]
255 = (FF)_16



## BasicPrimalityTest

**Input:** a natural number $n$. **Output:** `true` if $n$ is prime, `false` otherwise.

A divisor $d < n$ of $n$ is at most $n/2$: it suffices to test the candidates $d = 2, 3, \dots, \lfloor n/2 \rfloor$. For a prime number the loop is executed completely: for $n = 29$ the candidates are 13. The function also returns the number of candidates tested.

In [4]:
def basic_primality_test(n, trace=False):
    tested = 0                          # candidates tested
    if n < 2:
        if trace:
            print("%3s %8s  %s" % ("-", "-", "n < 2: return false"))
        return False, tested            # 0 and 1 are not prime
    for d in range(2, n // 2 + 1):      # d = 2..⌊n/2⌋
        tested += 1
        if n % d == 0:
            if trace:
                print("%3d %8d  %s" % (d, n % d, "return false: d divides n"))
            return False, tested        # d divides n: not prime
        if trace:
            print("%3d %8d  %s" % (d, n % d, "continue"))
    if trace:
        print("%3s %8s  %s" % ("-", "-", "the loop ends: return true"))
    return True, tested                 # no divisor found: n is prime


### Execution on the examples of the chapter

In [5]:
def run(n):
    print("Check: n = %d, floor(n/2) = %d" % (n, n // 2))
    print("%3s %8s  %s" % ("d", "n mod d", "action"))
    result, tested = basic_primality_test(n, trace=True)
    print("Return: %s, candidates tested: %d" % (str(result).lower(), tested))
    print()


run(36)
run(29)
run(25)

Check: n = 36, floor(n/2) = 18
  d  n mod d  action
  2        0  return false: d divides n
Return: false, candidates tested: 1

Check: n = 29, floor(n/2) = 14
  d  n mod d  action
  2        1  continue
  3        2  continue
  4        1  continue
  5        4  continue
  6        5  continue
  7        1  continue
  8        5  continue
  9        2  continue
 10        9  continue
 11        7  continue
 12        5  continue
 13        3  continue
 14        1  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 13

Check: n = 25, floor(n/2) = 12
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        0  return false: d divides n
Return: false, candidates tested: 4



### Try it yourself

Change the input in the cell below and run it again. The number to check is $n$.

In [6]:
run(97)

Check: n = 97, floor(n/2) = 48
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        2  continue
  6        1  continue
  7        6  continue
  8        1  continue
  9        7  continue
 10        7  continue
 11        9  continue
 12        1  continue
 13        6  continue
 14       13  continue
 15        7  continue
 16        1  continue
 17       12  continue
 18        7  continue
 19        2  continue
 20       17  continue
 21       13  continue
 22        9  continue
 23        5  continue
 24        1  continue
 25       22  continue
 26       19  continue
 27       16  continue
 28       13  continue
 29       10  continue
 30        7  continue
 31        4  continue
 32        1  continue
 33       31  continue
 34       29  continue
 35       27  continue
 36       25  continue
 37       23  continue
 38       21  continue
 39       19  continue
 40       17  continue
 41       15  continue
 42       13  continue
 43  

## PrimalityTest

**Input:** a natural number $n$. **Output:** `true` if $n$ is prime, `false` otherwise.

The divisors come in pairs $d$ and $n/d$ and in each pair one of the two numbers is at most $\sqrt{n}$: a number $n \ge 2$ is prime if and only if no number in $\{2, 3, \dots, \lfloor \sqrt{n} \rfloor\}$ divides it. For $n = 2$ and $n = 3$ the loop is not executed and the function returns `true`. For $n = 29$ the candidates are 4, instead of the 13 of the basic algorithm.

In [7]:
from math import isqrt                 # integer part of the square root, exact on integers


def primality_test(n, trace=False):
    tested = 0                          # candidates tested
    if n < 2:
        if trace:
            print("%3s %8s  %s" % ("-", "-", "n < 2: return false"))
        return False, tested            # 0 and 1 are not prime
    d = 2
    while d <= isqrt(n):                # d <= ⌊√n⌋
        tested += 1
        if n % d == 0:
            if trace:
                print("%3d %8d  %s" % (d, n % d, "return false: d divides n"))
            return False, tested        # d divides n: not prime
        if trace:
            print("%3d %8d  %s" % (d, n % d, "continue"))
        d = d + 1
    if trace:
        print("%3s %8s  %s" % ("-", "-", "the loop ends: return true"))
    return True, tested                 # no divisor found: n is prime


### Execution on the examples of the chapter

In [8]:
def run(n):
    print("Check: n = %d, floor(sqrt(n)) = %d" % (n, isqrt(n)))
    print("%3s %8s  %s" % ("d", "n mod d", "action"))
    result, tested = primality_test(n, trace=True)
    print("Return: %s, candidates tested: %d" % (str(result).lower(), tested))
    print()


run(36)
run(29)
run(25)
run(2)       # the loop is not executed

Check: n = 36, floor(sqrt(n)) = 6
  d  n mod d  action
  2        0  return false: d divides n
Return: false, candidates tested: 1

Check: n = 29, floor(sqrt(n)) = 5
  d  n mod d  action
  2        1  continue
  3        2  continue
  4        1  continue
  5        4  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 4

Check: n = 25, floor(sqrt(n)) = 5
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        0  return false: d divides n
Return: false, candidates tested: 4

Check: n = 2, floor(sqrt(n)) = 1
  d  n mod d  action
  -        -  the loop ends: return true
Return: true, candidates tested: 0



### Try it yourself

Change the input in the cell below and run it again. The number to check is $n$.

In [9]:
run(97)

Check: n = 97, floor(sqrt(n)) = 9
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        2  continue
  6        1  continue
  7        6  continue
  8        1  continue
  9        7  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 8



## SieveOfEratosthenes

**Input:** a natural number $n \ge 2$. **Output:** a boolean array $prime[1..n]$ where $prime[i] = 1$ if and only if $i$ is prime.

We start with all the numbers marked as prime (except 1) and, for every prime $p \le \lfloor \sqrt{n} \rfloor$, we mark its multiples as composite starting from $p^2$. In Python, C++ and Java the array starts from 0: `prime[i]` is associated with the number $i$, so it has $n+1$ entries `prime[0..n]` and `prime[0]` (the number 0, not prime) is 0.

In [10]:
from math import isqrt


def sieve_of_eratosthenes(n, trace=False):
    prime = [1] * (n + 1)               # prime[0..n], all set to 1
    prime[0] = 0                        # 0 is not prime
    prime[1] = 0                        # 1 is not prime
    if trace:
        print("%5s  %s" % ("start", " ".join(map(str, prime))))
    p = 2
    while p <= isqrt(n):                # p <= ⌊√n⌋
        js = []                         # the values of j (only for the trace)
        if prime[p] == 1:
            j = p * p                   # start from p^2
            while j <= n:
                prime[j] = 0            # j is not prime
                js.append(j)
                j = j + p
        if trace:
            print("%5d  %s  %s" % (p, " ".join(map(str, prime)),
                  "j = " + ", ".join(map(str, js)) if js else "no inner loop: prime[%d] is 0" % p))
        p = p + 1
    return prime


### Execution on the examples of the chapter

In [11]:
def run(n):
    print("Sieve: n = %d, floor(sqrt(n)) = %d" % (n, isqrt(n)))
    print("%5s  %s" % ("p", "prime[0..%d]" % n))
    prime = sieve_of_eratosthenes(n, trace=True)
    print("Prime numbers <= %d: %s" % (n, [i for i in range(n + 1) if prime[i] == 1]))
    print()


run(10)
run(20)

Sieve: n = 10, floor(sqrt(n)) = 3
    p  prime[0..10]
start  0 0 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10
    3  0 0 1 1 0 1 0 1 0 0 0  j = 9
Prime numbers <= 10: [2, 3, 5, 7]

Sieve: n = 20, floor(sqrt(n)) = 4
    p  prime[0..20]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  j = 9, 12, 15, 18
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  no inner loop: prime[4] is 0
Prime numbers <= 20: [2, 3, 5, 7, 11, 13, 17, 19]



### Try it yourself

Change the input in the cell below and run it again. The number is $n$ (at least 2).

In [12]:
run(30)

Sieve: n = 30, floor(sqrt(n)) = 5
    p  prime[0..30]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  j = 9, 12, 15, 18, 21, 24, 27, 30
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  no inner loop: prime[4] is 0
    5  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0  j = 25, 30
Prime numbers <= 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]



## EuclideanAlgorithm

**Input:** two natural numbers $n, m \ge 1$. **Output:** $\gcd(n, m)$, the greatest common divisor.

The pair $(n, m)$ is replaced with $(m,\ n \bmod m)$, which has the same common divisors, until the second number is 0: then $\gcd(n, 0) = n$. The parameters are passed by value: the function modifies its own copies. Two numbers are coprime if and only if the result is 1.

In [13]:
def euclidean_algorithm(n, m, trace=False):
    step = 0                            # only for the trace
    while m != 0:
        step += 1
        r = n % m
        if trace:
            print("%5d %5d %5d  %d mod %d = %d" % (step, n, m, n, m, r))
        n = m
        m = r
    if trace:
        print("%5d %5d %5d  %s" % (step + 1, n, m, "--- (the loop ends)"))
    return n                            # m = 0: n is the GCD


### Execution on the examples of the chapter

In [14]:
def run(n, m):
    print("GCD computation: n = %d, m = %d" % (n, m))
    print("%5s %5s %5s  %s" % ("step", "n", "m", "r = n mod m"))
    g = euclidean_algorithm(n, m, trace=True)
    print("Return: gcd(%d, %d) = %d" % (n, m, g))
    print("coprime: %s" % str(g == 1).lower())
    print()


run(48, 18)
run(252, 105)
run(35, 12)

GCD computation: n = 48, m = 18
 step     n     m  r = n mod m
    1    48    18  48 mod 18 = 12
    2    18    12  18 mod 12 = 6
    3    12     6  12 mod 6 = 0
    4     6     0  --- (the loop ends)
Return: gcd(48, 18) = 6
coprime: false

GCD computation: n = 252, m = 105
 step     n     m  r = n mod m
    1   252   105  252 mod 105 = 42
    2   105    42  105 mod 42 = 21
    3    42    21  42 mod 21 = 0
    4    21     0  --- (the loop ends)
Return: gcd(252, 105) = 21
coprime: false

GCD computation: n = 35, m = 12
 step     n     m  r = n mod m
    1    35    12  35 mod 12 = 11
    2    12    11  12 mod 11 = 1
    3    11     1  11 mod 1 = 0
    4     1     0  --- (the loop ends)
Return: gcd(35, 12) = 1
coprime: true



### Try it yourself

Change the input in the cell below and run it again. The two numbers are $n$ and $m$.

In [15]:
run(1071, 462)

GCD computation: n = 1071, m = 462
 step     n     m  r = n mod m
    1  1071   462  1071 mod 462 = 147
    2   462   147  462 mod 147 = 21
    3   147    21  147 mod 21 = 0
    4    21     0  --- (the loop ends)
Return: gcd(1071, 462) = 21
coprime: false



---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).